# Challenge 09 Colab Ultra: SVM with preprocessing search

Esta variante asume que el espacio util de `SVM` ya esta bastante claro.

Por eso el objetivo ya no es otro grid search enorme de `C` y `gamma`, sino evaluar si el salto viene de:

- un mejor escalado
- una limpieza suave de instancias
- una mejor transformacion de la distribucion de las variables

## Estrategia

1. `Stage 1`: screening amplio de pipelines de preprocesamiento + SVM.
2. `Stage 2`: shortlist con `3-fold CV`.
3. `Stage 3`: refinamiento local solo alrededor de la mejor region.
4. Modelo final, submission y artefactos para stacking.

## 0. Imports and global constants

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import gc
import itertools
import json
import platform
import time
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from sklearn.model_selection import ParameterSampler, StratifiedKFold, train_test_split
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import PowerTransformer, QuantileTransformer, RobustScaler, StandardScaler
from sklearn.svm import SVC

try:
    import psutil
except ImportError:
    psutil = None

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["savefig.bbox"] = "tight"

RANDOM_STATE = 301655
VALID_SIZE = 0.20
NOTEBOOK_SLUG = "challenge_09_svm_preprocessing_colab_ultra"

## 1. Create the Colab workspace

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import files  # type: ignore
else:
    files = None

if IN_COLAB:
    WORKSPACE_ROOT = Path("/content/challenge_svm_preprocessing_ultra_workspace")
else:
    cwd = Path.cwd().resolve()
    if (cwd / "challenge" / "data" / "training.csv").exists():
        WORKSPACE_ROOT = cwd / "challenge"
    elif (cwd / "data" / "training.csv").exists():
        WORKSPACE_ROOT = cwd
    else:
        WORKSPACE_ROOT = cwd / "challenge_svm_preprocessing_ultra_workspace"

DATA_DIR = WORKSPACE_ROOT / "data"
TRAIN_PATH = DATA_DIR / "training.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_PATH = DATA_DIR / "sample.csv"

OUTPUT_ROOT = WORKSPACE_ROOT / "output"
PERSIST_ROOT = OUTPUT_ROOT / NOTEBOOK_SLUG
CHECKPOINT_DIR = PERSIST_ROOT / "checkpoints"
SUBMISSION_DIR = WORKSPACE_ROOT / "submissions"
EXPORT_DIR = WORKSPACE_ROOT / "exports"

for path in [WORKSPACE_ROOT, DATA_DIR, OUTPUT_ROOT, PERSIST_ROOT, CHECKPOINT_DIR, SUBMISSION_DIR, EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("WORKSPACE_ROOT:", WORKSPACE_ROOT)
print("PERSIST_ROOT:", PERSIST_ROOT)

## 2. Inspect the workspace and expected files

In [ ]:
expected_files = {
    "training.csv": TRAIN_PATH,
    "test.csv": TEST_PATH,
    "sample.csv": SAMPLE_PATH,
}

print("Workspace directories:")
for path in [WORKSPACE_ROOT, DATA_DIR, OUTPUT_ROOT, PERSIST_ROOT, CHECKPOINT_DIR, SUBMISSION_DIR, EXPORT_DIR]:
    print("-", path)

print("\nData file status:")
for filename, path in expected_files.items():
    print(f"- {filename}: {'OK' if path.exists() else 'MISSING'} -> {path}")

## 3. Optional: upload the CSV files manually

In [ ]:
UPLOAD_DATA_FILES = False

if UPLOAD_DATA_FILES:
    if not IN_COLAB:
        raise RuntimeError("This upload helper is intended for Google Colab.")

    uploaded = files.upload()
    for original_name, file_bytes in uploaded.items():
        filename = Path(original_name).name
        target_path = DATA_DIR / filename
        target_path.write_bytes(file_bytes)
        print("Saved:", target_path)
else:
    print("Set UPLOAD_DATA_FILES = True if you want to upload training.csv, test.csv and sample.csv.")

## 4. Optional: restore resume bundles

In [ ]:
RESTORE_RESUME_BUNDLE = False

if RESTORE_RESUME_BUNDLE:
    if not IN_COLAB:
        raise RuntimeError("This restore helper is intended for Google Colab.")

    uploaded = files.upload()
    zip_names = [Path(name).name for name in uploaded if str(name).lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one ZIP file when restoring a resume bundle.")

    bundle_name = zip_names[0]
    bundle_path = EXPORT_DIR / bundle_name
    bundle_path.write_bytes(uploaded[bundle_name])
    with zipfile.ZipFile(bundle_path, "r") as zip_file:
        zip_file.extractall(WORKSPACE_ROOT)
    print("Resume bundle restored into:", WORKSPACE_ROOT)
else:
    print("Set RESTORE_RESUME_BUNDLE = True if you want to restore a previous ZIP bundle.")

## 5. Checkpoint helpers and reusable utilities

In [ ]:
DEFAULT_BUNDLE_NAME = "challenge_09_svm_preprocessing_colab_ultra_resume.zip"


def save_current_figure(filename: str) -> Path:
    path = PERSIST_ROOT / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()
    return path


def write_json_atomic(path: Path, payload: dict | list) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(payload, indent=2))
    tmp_path.replace(path)


def save_dataframe_atomic(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp_path, index=False)
    tmp_path.replace(path)


def read_dataframe(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def normalize_value(value):
    if isinstance(value, np.generic):
        return value.item()
    return value


def candidate_signature(params: dict) -> str:
    normalized = {key: normalize_value(value) for key, value in params.items()}
    return json.dumps(normalized, sort_keys=True)


def update_manifest(extra_payload: dict) -> dict:
    manifest_path = CHECKPOINT_DIR / "manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    manifest.update(extra_payload)
    write_json_atomic(manifest_path, manifest)
    return manifest


def create_resume_bundle(bundle_name: str = DEFAULT_BUNDLE_NAME, include_data: bool = True) -> Path:
    bundle_path = EXPORT_DIR / bundle_name
    if bundle_path.exists():
        bundle_path.unlink()

    paths_to_pack = []
    if include_data:
        paths_to_pack.append(DATA_DIR)
    paths_to_pack.extend([OUTPUT_ROOT, SUBMISSION_DIR])

    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
        for root_path in paths_to_pack:
            if not root_path.exists():
                continue
            for nested in root_path.rglob("*"):
                if nested.is_dir():
                    continue
                relative_path = nested.relative_to(WORKSPACE_ROOT)
                zip_file.write(nested, arcname=str(relative_path))

    return bundle_path


def checkpoint_housekeeping(stage_name: str, *, refresh_bundle: bool = False, include_data_in_bundle: bool = False) -> dict:
    payload = {"last_checkpoint_stage": stage_name}
    if refresh_bundle:
        bundle_path = create_resume_bundle(include_data=include_data_in_bundle)
        payload["resume_bundle_path"] = str(bundle_path)
        payload["resume_bundle_size_mb"] = round(bundle_path.stat().st_size / (1024 ** 2), 3)
    update_manifest(payload)
    return payload

## 6. Strategy-specific helpers

In [ ]:
CLEANING_METHODS = ["none", "lof_0.03", "lof_0.05"]
SCALER_METHODS = ["standard", "robust", "quantile_normal", "power_yeo"]


def normalize_candidate(params: dict) -> dict:
    normalized = {key: normalize_value(value) for key, value in params.items()}
    normalized["clean__method"] = str(normalized["clean__method"])
    normalized["scale__method"] = str(normalized["scale__method"])
    normalized["pca__n_components"] = int(normalized["pca__n_components"])
    normalized["model__C"] = float(normalized["model__C"])
    normalized["model__gamma"] = float(normalized["model__gamma"])
    return normalized


def candidate_record(candidate_id: str, params: dict, source: str) -> dict:
    params = normalize_candidate(params)
    return {
        "candidate_id": candidate_id,
        "source": source,
        "params": params,
        "signature": candidate_signature(params),
    }


def candidate_to_row(candidate: dict) -> dict:
    params = candidate["params"]
    return {
        "candidate_id": candidate["candidate_id"],
        "source": candidate["source"],
        "signature": candidate["signature"],
        "clean__method": params["clean__method"],
        "scale__method": params["scale__method"],
        "pca__n_components": params["pca__n_components"],
        "model__C": params["model__C"],
        "model__gamma": params["model__gamma"],
        "params_json": json.dumps(params, sort_keys=True),
    }


def apply_lof_cleaning(X_input: np.ndarray, y_input: np.ndarray, contamination: float) -> tuple[np.ndarray, np.ndarray, int]:
    scaled = StandardScaler().fit_transform(X_input)
    keep_mask = np.ones(len(X_input), dtype=bool)
    for cls in np.unique(y_input):
        idx = np.where(y_input == cls)[0]
        if len(idx) < 12:
            continue
        n_neighbors = min(25, len(idx) - 1)
        lof = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
        pred = lof.fit_predict(scaled[idx])
        keep_mask[idx] = pred == 1
    return X_input[keep_mask], y_input[keep_mask], int((~keep_mask).sum())


def apply_cleaning(X_input: np.ndarray, y_input: np.ndarray, method: str) -> tuple[np.ndarray, np.ndarray, dict]:
    if method == "none":
        return X_input, y_input, {"removed_rows": 0}
    contamination = float(method.split("_")[1])
    X_clean, y_clean, removed = apply_lof_cleaning(X_input, y_input, contamination)
    return X_clean, y_clean, {"removed_rows": removed}


def choose_scaler(name: str, n_samples: int):
    if name == "standard":
        return StandardScaler()
    if name == "robust":
        return RobustScaler()
    if name == "quantile_normal":
        return QuantileTransformer(n_quantiles=min(1000, max(10, n_samples)), output_distribution="normal", random_state=RANDOM_STATE)
    if name == "power_yeo":
        return PowerTransformer(method="yeo-johnson")
    raise ValueError(f"Unknown scaler: {name}")


def build_svm(candidate: dict, probability: bool = False) -> SVC:
    params = candidate["params"]
    return SVC(
        kernel="rbf",
        C=float(params["model__C"]),
        gamma=float(params["model__gamma"]),
        cache_size=SVM_CACHE_MB,
        shrinking=True,
        probability=probability,
    )


def fit_bundle(X_fit: np.ndarray, y_fit: np.ndarray, candidate: dict, probability: bool = False) -> dict:
    params = candidate["params"]
    X_clean, y_clean, cleaning_info = apply_cleaning(X_fit, y_fit, params["clean__method"])
    scaler = choose_scaler(params["scale__method"], len(X_clean))
    X_clean_scaled = scaler.fit_transform(X_clean)
    n_components = min(int(params["pca__n_components"]), X_clean_scaled.shape[0], X_clean_scaled.shape[1])
    pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
    X_clean_proj = pca.fit_transform(X_clean_scaled).astype(np.float32)
    model = build_svm(candidate, probability=probability)
    model.fit(X_clean_proj, y_clean)
    return {
        "candidate": candidate,
        "scaler": scaler,
        "pca": pca,
        "model": model,
        "cleaning_info": cleaning_info,
    }


def predict_bundle(bundle: dict, X_input: np.ndarray, proba: bool = False) -> np.ndarray:
    X_scaled = bundle["scaler"].transform(X_input)
    X_proj = bundle["pca"].transform(X_scaled).astype(np.float32)
    if proba:
        return bundle["model"].predict_proba(X_proj)
    return bundle["model"].predict(X_proj)


def evaluate_candidate(X_fit: np.ndarray, y_fit: np.ndarray, X_eval: np.ndarray, y_eval: np.ndarray, candidate: dict) -> tuple[float, float, int]:
    start = time.time()
    bundle = fit_bundle(X_fit, y_fit, candidate, probability=False)
    predictions = predict_bundle(bundle, X_eval, proba=False)
    accuracy = accuracy_score(y_eval, predictions)
    elapsed = round(time.time() - start, 3)
    removed_rows = int(bundle["cleaning_info"].get("removed_rows", 0))
    return float(accuracy), elapsed, removed_rows


def refresh_summary_from_folds(fold_path: Path, summary_path: Path, candidates: list[dict], n_splits: int, stage_name: str) -> pd.DataFrame:
    fold_df = read_dataframe(fold_path)
    if fold_df.empty:
        return pd.DataFrame()

    summary_rows = []
    for candidate in candidates:
        candidate_fold_df = fold_df[fold_df["candidate_id"] == candidate["candidate_id"]]
        if candidate_fold_df["fold_idx"].nunique() < n_splits:
            continue
        row = {
            **candidate_to_row(candidate),
            "stage": stage_name,
            "cv_mean_accuracy": float(candidate_fold_df["fold_accuracy"].mean()),
            "cv_std_accuracy": float(candidate_fold_df["fold_accuracy"].std(ddof=0)),
            "mean_removed_rows": float(candidate_fold_df["removed_rows"].mean()),
            "total_fit_seconds": float(candidate_fold_df["fit_seconds"].sum()),
            "completed_folds": int(candidate_fold_df["fold_idx"].nunique()),
        }
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty:
        summary_df = summary_df.sort_values(["cv_mean_accuracy", "mean_removed_rows", "total_fit_seconds"], ascending=[False, True, True]).reset_index(drop=True)
        save_dataframe_atomic(summary_df, summary_path)
    return summary_df

## 7. Validate that the required CSV files are present

In [ ]:
missing_files = [str(path) for path in [TRAIN_PATH, TEST_PATH, SAMPLE_PATH] if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required data files. Upload training.csv, test.csv and sample.csv first or restore a resume ZIP. "
        f"Missing: {missing_files}"
    )

print("All required CSV files are present.")

## 8. Runtime inspection and search budget

In [ ]:
CPU_COUNT = os.cpu_count() or 2
RAM_GB = None if psutil is None else round(psutil.virtual_memory().total / (1024 ** 3), 2)
SVM_CACHE_MB = 1024 if (RAM_GB is not None and RAM_GB >= 20) else 512

SEARCH_PRESETS = {
    "balanced": {
        "stage1_n_iter": 64,
        "stage1_pool_fraction": 0.70,
        "stage1_eval_size": 0.25,
        "stage2_top_k": 14,
        "stage2_cv": 3,
        "stage3_seed_top_k": 3,
        "stage3_cv": 5,
    },
    "aggressive": {
        "stage1_n_iter": 96,
        "stage1_pool_fraction": 0.82,
        "stage1_eval_size": 0.25,
        "stage2_top_k": 18,
        "stage2_cv": 3,
        "stage3_seed_top_k": 4,
        "stage3_cv": 5,
    },
}

SEARCH_PROFILE = "aggressive"
PROFILE = SEARCH_PRESETS[SEARCH_PROFILE]

RUN_STAGE_1 = True
RUN_STAGE_2 = True
RUN_STAGE_3 = True
TRAIN_FINAL_MODEL = True

print(
    {
        "python": platform.python_version(),
        "sklearn": sklearn.__version__,
        "cpu_count": CPU_COUNT,
        "ram_gb": RAM_GB,
        "svm_cache_mb": SVM_CACHE_MB,
        "search_profile": SEARCH_PROFILE,
    }
)

## 9. Load the challenge data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_df = pd.read_csv(SAMPLE_PATH)

feature_names = [column for column in train_df.columns if column not in {"id", "class"}]
train_ids = train_df["id"].to_numpy()
test_ids = test_df["id"].to_numpy()

X_full = train_df[feature_names].astype(np.float32).to_numpy()
y_full = train_df["class"].astype(np.int8).to_numpy()
X_test_full = test_df[feature_names].astype(np.float32).to_numpy()

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Class balance:", train_df["class"].value_counts().sort_index().to_dict())
print("Missing values in training:", int(train_df.isna().sum().sum()))
print("Duplicated rows in training:", int(train_df.duplicated().sum()))

X_train, X_valid, y_train, y_valid = train_test_split(
    X_full,
    y_full,
    test_size=VALID_SIZE,
    stratify=y_full,
    random_state=RANDOM_STATE,
)

print("Train split:", X_train.shape, y_train.shape)
print("Validation split:", X_valid.shape, y_valid.shape)

## 10. Stage 1

In [ ]:
STAGE1_SPACE = {
    "clean__method": CLEANING_METHODS,
    "scale__method": SCALER_METHODS,
    "pca__n_components": [96, 104, 112, 120, 128, 136, 144],
    "model__C": [2.0, 3.0, 4.0, 6.0, 8.0],
    "model__gamma": [0.005, 0.01, 0.02],
}

stage1_candidates_path = CHECKPOINT_DIR / "stage1_candidates.json"
stage1_results_path = CHECKPOINT_DIR / "stage1_holdout_results.csv"


def get_or_create_stage1_candidates() -> list[dict]:
    if stage1_candidates_path.exists():
        return json.loads(stage1_candidates_path.read_text())

    sampler = ParameterSampler(STAGE1_SPACE, n_iter=PROFILE["stage1_n_iter"], random_state=RANDOM_STATE)
    candidates = []
    seen = set()
    for sampled_params in sampler:
        normalized = normalize_candidate(sampled_params)
        signature = candidate_signature(normalized)
        if signature in seen:
            continue
        seen.add(signature)
        candidates.append(candidate_record(f"stage1_{len(candidates):03d}", normalized, "stage1_random_holdout"))
    write_json_atomic(stage1_candidates_path, candidates)
    return candidates


X_stage1_pool, _, y_stage1_pool, _ = train_test_split(
    X_train,
    y_train,
    train_size=PROFILE["stage1_pool_fraction"],
    stratify=y_train,
    random_state=RANDOM_STATE,
)
X_stage1_fit, X_stage1_eval, y_stage1_fit, y_stage1_eval = train_test_split(
    X_stage1_pool,
    y_stage1_pool,
    test_size=PROFILE["stage1_eval_size"],
    stratify=y_stage1_pool,
    random_state=RANDOM_STATE,
)

stage1_candidates = get_or_create_stage1_candidates()
stage1_results = read_dataframe(stage1_results_path)
completed_stage1 = set(stage1_results["candidate_id"]) if not stage1_results.empty else set()

print("Stage 1 candidates:", len(stage1_candidates))
print("Stage 1 completed:", len(completed_stage1))

if RUN_STAGE_1:
    for candidate in tqdm([c for c in stage1_candidates if c["candidate_id"] not in completed_stage1], desc="Stage 1 screening"):
        accuracy, elapsed, removed_rows = evaluate_candidate(X_stage1_fit, y_stage1_fit, X_stage1_eval, y_stage1_eval, candidate)
        row = {
            **candidate_to_row(candidate),
            "stage": "stage1_holdout",
            "holdout_accuracy": accuracy,
            "fit_seconds": elapsed,
            "removed_rows": removed_rows,
        }
        stage1_results = pd.concat([stage1_results, pd.DataFrame([row])], ignore_index=True) if not stage1_results.empty else pd.DataFrame([row])
        stage1_results = stage1_results.sort_values(["holdout_accuracy", "removed_rows", "fit_seconds"], ascending=[False, True, True]).reset_index(drop=True)
        save_dataframe_atomic(stage1_results, stage1_results_path)
        update_manifest(
            {
                "stage1_completed_candidates": int(stage1_results["candidate_id"].nunique()),
                "stage1_total_candidates": len(stage1_candidates),
                "stage1_best_holdout_accuracy": float(stage1_results["holdout_accuracy"].max()),
            }
        )
        gc.collect()

stage1_results = read_dataframe(stage1_results_path)
if not stage1_results.empty:
    display(stage1_results.head(12))
    checkpoint_housekeeping("stage1_complete", refresh_bundle=True, include_data_in_bundle=False)

## 11. Stage 2

In [ ]:
stage2_fold_path = CHECKPOINT_DIR / "stage2_cv_fold_results.csv"
stage2_summary_path = CHECKPOINT_DIR / "stage2_cv_summary.csv"

if stage1_results.empty:
    raise RuntimeError("Stage 1 produced no results.")

stage2_shortlist = (
    stage1_results.sort_values(["holdout_accuracy", "removed_rows", "fit_seconds"], ascending=[False, True, True])
    .drop_duplicates("signature")
    .head(PROFILE["stage2_top_k"])
    .copy()
)
stage2_candidates = [
    candidate_record(row["candidate_id"], json.loads(row["params_json"]), "stage2_shortlist_from_stage1")
    for row in stage2_shortlist.to_dict(orient="records")
]

stage2_cv = StratifiedKFold(n_splits=PROFILE["stage2_cv"], shuffle=True, random_state=RANDOM_STATE)
stage2_splits = list(stage2_cv.split(X_train, y_train))

fold_df = read_dataframe(stage2_fold_path)
completed_pairs = set(zip(fold_df["candidate_id"].astype(str), fold_df["fold_idx"].astype(int))) if not fold_df.empty else set()

print("Stage 2 candidates:", len(stage2_candidates))
print("Stage 2 completed fold-pairs:", len(completed_pairs))

if RUN_STAGE_2:
    for fold_idx, (fit_idx, eval_idx) in enumerate(tqdm(stage2_splits, desc="Stage 2 folds")):
        pending = [candidate for candidate in stage2_candidates if (candidate["candidate_id"], fold_idx) not in completed_pairs]
        for candidate in tqdm(pending, desc=f"Stage 2 fold {fold_idx}", leave=False):
            accuracy, elapsed, removed_rows = evaluate_candidate(
                X_train[fit_idx],
                y_train[fit_idx],
                X_train[eval_idx],
                y_train[eval_idx],
                candidate,
            )
            row = {
                **candidate_to_row(candidate),
                "stage": "stage2_cv",
                "fold_idx": fold_idx,
                "fold_accuracy": accuracy,
                "fit_seconds": elapsed,
                "removed_rows": removed_rows,
            }
            fold_df = pd.concat([fold_df, pd.DataFrame([row])], ignore_index=True) if not fold_df.empty else pd.DataFrame([row])
            save_dataframe_atomic(fold_df, stage2_fold_path)
            completed_pairs.add((candidate["candidate_id"], fold_idx))
            gc.collect()

        stage2_summary = refresh_summary_from_folds(stage2_fold_path, stage2_summary_path, stage2_candidates, PROFILE["stage2_cv"], "stage2_cv")
        if not stage2_summary.empty:
            update_manifest(
                {
                    "stage2_completed_candidates": int(stage2_summary["candidate_id"].nunique()),
                    "stage2_total_candidates": len(stage2_candidates),
                    "stage2_best_cv_accuracy": float(stage2_summary["cv_mean_accuracy"].max()),
                }
            )

stage2_summary = read_dataframe(stage2_summary_path)
if not stage2_summary.empty:
    display(stage2_summary.head(12))
    checkpoint_housekeeping("stage2_complete", refresh_bundle=True, include_data_in_bundle=False)

## 12. Stage 3

In [ ]:
stage3_candidates_path = CHECKPOINT_DIR / "stage3_local_candidates.json"
stage3_fold_path = CHECKPOINT_DIR / "stage3_local_cv_fold_results.csv"
stage3_summary_path = CHECKPOINT_DIR / "stage3_local_cv_summary.csv"

if stage2_summary.empty:
    raise RuntimeError("Stage 2 produced no results.")

stage3_seed_rows = (
    stage2_summary.sort_values(["cv_mean_accuracy", "mean_removed_rows", "total_fit_seconds"], ascending=[False, True, True])
    .drop_duplicates("signature")
    .head(PROFILE["stage3_seed_top_k"])
    .copy()
)


def get_or_create_stage3_candidates() -> list[dict]:
    if stage3_candidates_path.exists():
        return json.loads(stage3_candidates_path.read_text())

    seen = set()
    candidates = []
    next_index = 0
    for seed_rank, row in enumerate(stage3_seed_rows.to_dict(orient="records")):
        seed = json.loads(row["params_json"])
        pca_values = sorted({value for value in [seed["pca__n_components"] - 16, seed["pca__n_components"] - 8, seed["pca__n_components"], seed["pca__n_components"] + 8, seed["pca__n_components"] + 16] if 48 <= value <= X_train.shape[1]})
        c_values = sorted({round(max(0.5, seed["model__C"] * ratio), 4) for ratio in [0.75, 1.0, 1.5, 2.0]})
        gamma_values = sorted({round(max(0.001, seed["model__gamma"] * ratio), 5) for ratio in [0.5, 1.0, 1.5, 2.0] if max(0.001, seed["model__gamma"] * ratio) <= 0.05})
        clean_values = sorted({seed["clean__method"], "none", "lof_0.03", "lof_0.05"})
        scale_values = sorted({seed["scale__method"], "standard", "robust", "quantile_normal", "power_yeo"})
        for clean_method in clean_values:
            for scale_method in scale_values:
                for pca_components in pca_values:
                    for c_value in c_values:
                        for gamma_value in gamma_values:
                            params = normalize_candidate(
                                {
                                    "clean__method": clean_method,
                                    "scale__method": scale_method,
                                    "pca__n_components": pca_components,
                                    "model__C": c_value,
                                    "model__gamma": gamma_value,
                                }
                            )
                            signature = candidate_signature(params)
                            if signature in seen:
                                continue
                            seen.add(signature)
                            candidates.append(candidate_record(f"stage3_{next_index:03d}", params, f"stage3_seed_{seed_rank}"))
                            next_index += 1
    write_json_atomic(stage3_candidates_path, candidates)
    return candidates


stage3_candidates = get_or_create_stage3_candidates()
stage3_cv = StratifiedKFold(n_splits=PROFILE["stage3_cv"], shuffle=True, random_state=RANDOM_STATE)
stage3_splits = list(stage3_cv.split(X_train, y_train))

fold_df = read_dataframe(stage3_fold_path)
completed_pairs = set(zip(fold_df["candidate_id"].astype(str), fold_df["fold_idx"].astype(int))) if not fold_df.empty else set()

print("Stage 3 candidates:", len(stage3_candidates))
print("Stage 3 completed fold-pairs:", len(completed_pairs))

if RUN_STAGE_3:
    for fold_idx, (fit_idx, eval_idx) in enumerate(tqdm(stage3_splits, desc="Stage 3 folds")):
        pending = [candidate for candidate in stage3_candidates if (candidate["candidate_id"], fold_idx) not in completed_pairs]
        for candidate in tqdm(pending, desc=f"Stage 3 fold {fold_idx}", leave=False):
            accuracy, elapsed, removed_rows = evaluate_candidate(
                X_train[fit_idx],
                y_train[fit_idx],
                X_train[eval_idx],
                y_train[eval_idx],
                candidate,
            )
            row = {
                **candidate_to_row(candidate),
                "stage": "stage3_local_cv",
                "fold_idx": fold_idx,
                "fold_accuracy": accuracy,
                "fit_seconds": elapsed,
                "removed_rows": removed_rows,
            }
            fold_df = pd.concat([fold_df, pd.DataFrame([row])], ignore_index=True) if not fold_df.empty else pd.DataFrame([row])
            save_dataframe_atomic(fold_df, stage3_fold_path)
            completed_pairs.add((candidate["candidate_id"], fold_idx))
            gc.collect()

        stage3_summary = refresh_summary_from_folds(stage3_fold_path, stage3_summary_path, stage3_candidates, PROFILE["stage3_cv"], "stage3_local_cv")
        if not stage3_summary.empty:
            update_manifest(
                {
                    "stage3_completed_candidates": int(stage3_summary["candidate_id"].nunique()),
                    "stage3_total_candidates": len(stage3_candidates),
                    "stage3_best_cv_accuracy": float(stage3_summary["cv_mean_accuracy"].max()),
                }
            )

stage3_summary = read_dataframe(stage3_summary_path)
if not stage3_summary.empty:
    display(stage3_summary.head(12))
    checkpoint_housekeeping("stage3_complete", refresh_bundle=True, include_data_in_bundle=False)

## 13. Diagnostic plots

In [ ]:
if not stage1_results.empty:
    plt.figure(figsize=(12, 6))
    stage1_plot = stage1_results.groupby("scale__method", as_index=False)["holdout_accuracy"].max().sort_values("holdout_accuracy", ascending=False)
    sns.barplot(data=stage1_plot, x="scale__method", y="holdout_accuracy")
    plt.xticks(rotation=25, ha="right")
    plt.title("Stage 1 best holdout accuracy by preprocessing scaler")
    save_current_figure("stage1_scaler_methods.png")
    display(stage1_plot)

final_stage_df = stage3_summary if not stage3_summary.empty else stage2_summary if "stage2_summary" in globals() else stage1_results
if final_stage_df is not None and not final_stage_df.empty:
    plt.figure(figsize=(10, 7))
    plot_df = final_stage_df.head(20).copy()
    sns.scatterplot(data=plot_df, x="pca__n_components", y="cv_mean_accuracy" if "cv_mean_accuracy" in plot_df.columns else "holdout_accuracy", hue="scale__method", style="clean__method", s=120)
    plt.title("Top SVM preprocessing candidates")
    save_current_figure("top_candidates.png")

## 14. Final model, OOF artifacts and submission

In [ ]:
final_summary_df = stage3_summary if not stage3_summary.empty else stage2_summary if not stage2_summary.empty else stage1_results
if final_summary_df.empty:
    raise RuntimeError("No final candidate table is available.")

best_row = final_summary_df.iloc[0].to_dict()
best_candidate = candidate_record(best_row["candidate_id"], json.loads(best_row["params_json"]), "final_selection")

final_bundle = fit_bundle(X_train, y_train, best_candidate, probability=False)
valid_predictions = predict_bundle(final_bundle, X_valid, proba=False)
validation_accuracy = float(accuracy_score(y_valid, valid_predictions))

cm = confusion_matrix(y_valid, valid_predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", colorbar=False)
plt.title(f"Validation confusion matrix - accuracy={validation_accuracy:.4f}")
save_current_figure("validation_confusion_matrix.png")

oof_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_prob_1 = np.zeros(len(X_full), dtype=np.float32)
test_prob_1 = np.zeros(len(X_test_full), dtype=np.float32)
fold_records = []

for fold_idx, (fit_idx, eval_idx) in enumerate(tqdm(oof_cv.split(X_full, y_full), desc="OOF for stacking")):
    fold_bundle = fit_bundle(X_full[fit_idx], y_full[fit_idx], best_candidate, probability=True)
    oof_proba = predict_bundle(fold_bundle, X_full[eval_idx], proba=True)[:, 1]
    test_proba = predict_bundle(fold_bundle, X_test_full, proba=True)[:, 1]
    oof_prob_1[eval_idx] = oof_proba.astype(np.float32)
    test_prob_1 += test_proba.astype(np.float32) / oof_cv.n_splits
    fold_records.append(
        {
            "fold_idx": fold_idx,
            "fold_accuracy": float(accuracy_score(y_full[eval_idx], (oof_proba >= 0.5).astype(int))),
        }
    )
    gc.collect()

final_fit_bundle = fit_bundle(X_full, y_full, best_candidate, probability=False)
final_test_pred = predict_bundle(final_fit_bundle, X_test_full, proba=False).astype(int)

oof_df = pd.DataFrame(
    {
        "id": train_ids,
        "y_true": y_full.astype(int),
        "prob_1": oof_prob_1,
        "pred": (oof_prob_1 >= 0.5).astype(int),
        "source_model": NOTEBOOK_SLUG,
    }
)
test_prob_df = pd.DataFrame(
    {
        "id": test_ids,
        "prob_1": test_prob_1,
        "pred": (test_prob_1 >= 0.5).astype(int),
        "source_model": NOTEBOOK_SLUG,
    }
)
submission_df = sample_df.copy()
submission_df["class"] = final_test_pred.astype(int)

oof_path = PERSIST_ROOT / "oof_probabilities.csv"
test_prob_path = PERSIST_ROOT / "test_probabilities.csv"
fold_summary_path = PERSIST_ROOT / "oof_fold_summary.csv"
submission_path = SUBMISSION_DIR / "challenge_09_svm_preprocessing_colab_ultra_submission.csv"
summary_path = PERSIST_ROOT / "summary.json"

save_dataframe_atomic(oof_df, oof_path)
save_dataframe_atomic(test_prob_df, test_prob_path)
save_dataframe_atomic(pd.DataFrame(fold_records), fold_summary_path)
submission_df.to_csv(submission_path, index=False)

summary_payload = {
    "model_name": "SVM with preprocessing search",
    "model_key": "svm_preprocessing",
    "notebook_slug": NOTEBOOK_SLUG,
    "strategy": "colab_ultra_preprocessing_search",
    "search_profile": SEARCH_PROFILE,
    "best_stage": str(best_row.get("stage", "unknown")),
    "best_params": best_candidate["params"],
    "validation_accuracy": validation_accuracy,
    "validation_confusion_matrix": cm.tolist(),
    "oof_accuracy": float(accuracy_score(y_full, (oof_prob_1 >= 0.5).astype(int))),
    "oof_path": str(oof_path),
    "test_probability_path": str(test_prob_path),
    "submission_path": str(submission_path),
    "workspace_root": str(WORKSPACE_ROOT),
    "persist_root": str(PERSIST_ROOT),
}
write_json_atomic(summary_path, summary_payload)
checkpoint_housekeeping("final_model_complete", refresh_bundle=True, include_data_in_bundle=False)

print("Best params:", json.dumps(best_candidate["params"], indent=2))
print("Validation accuracy:", validation_accuracy)
print("OOF accuracy:", summary_payload["oof_accuracy"])
print("Submission path:", submission_path)

## 15. Optional: export and download a manual resume bundle

In [ ]:
INCLUDE_DATA_IN_MANUAL_BUNDLE = True
DOWNLOAD_BUNDLE_NOW = False

bundle_path = create_resume_bundle(include_data=INCLUDE_DATA_IN_MANUAL_BUNDLE)
print("Resume bundle saved to:", bundle_path)

if DOWNLOAD_BUNDLE_NOW and IN_COLAB:
    files.download(str(bundle_path))

## Notes

Esta notebook esta diseñada para responder una pregunta especifica:

> En SVM, ya no conviene seguir abriendo el espacio de `C/gamma`; conviene probar mejor preprocesamiento.

Tambien deja listos los artefactos para stacking:

- `oof_probabilities.csv`
- `test_probabilities.csv`